In [1]:
import os
os.chdir("../")

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    ALL_REQUIRED_FILES: list

In [3]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        
        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            ALL_REQUIRED_FILES=config.ALL_REQUIRED_FILES,
        )
        return data_validation_config

In [4]:
from textSummarizer.logging import logger

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_files_exist(self) -> bool:
        try:
            validation_status = None
            all_files = os.listdir(os.path.join("artifacts", "data_ingestion", "samsum_dataset"))
            for file in all_files:
                if file not in self.config.ALL_REQUIRED_FILES:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {str(validation_status)}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {str(validation_status)}")
            return validation_status
        except Exception as e:
            logger.error(f"Error occurred during validation: {e}")
            return False

        # status = True
        # for file in self.config.ALL_REQUIRED_FILES:
        #     file_path = os.path.join(self.config.root_dir, file)
        #     if not os.path.exists(file_path):
        #         logger.info(f"File {file} is missing.")
        #         status = False
        # return status

    # def run_validation(self) -> None:
    #     status = self.validate_all_files_exist()
    #     with open(self.config.STATUS_FILE, 'w') as f:
    #         f.write(str(status))

In [5]:
try:
    config_manager = ConfigurationManager()
    data_validation_config = config_manager.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_files_exist()
except Exception as e:
    raise e

[2025-05-03 09:32:40,067 : INFO : common : YAML file config\config.yaml loaded successfully.]
[2025-05-03 09:32:40,071 : INFO : common : YAML file params.yaml loaded successfully.]
[2025-05-03 09:32:40,073 : INFO : common : Directory artifacts created successfully.]
[2025-05-03 09:32:40,075 : INFO : common : Directory artifacts/data_validation created successfully.]
